1st clone repo to working directory

In [90]:
import git

repo = git.Repo.clone_from(
    "https://github.com/frankmollard/fuzzyfier",
    "./fuzzyfier"
)

In [91]:
import sys

2nd append path to environmental variables

In [92]:
sys.path.append("./fuzzyfier/src")

In [93]:
import numpy as np 
from fuzzyfication import fuzzyfication

In this example, the logical links deliver the decision directly.

In [31]:
stock = {"small": 0, "middle": 100, "high": 1000}
marketTrend= {"negative": -2, "stagnation": 0, "positive": 2}

In [75]:
fS=fuzzyfication(stock, 200)
fMT=fuzzyfication(marketTrend, 1)

Checking the activations of both values 101 and 1.25

In [76]:
fS.fuzzy()

{'small': np.float64(0.0),
 'middle': np.float64(0.888888888888889),
 'high': np.float64(0.1111111111111111)}

In [77]:
fMT.fuzzy()

{'negative': np.float64(0.0),
 'stagnation': np.float64(0.5),
 'positive': np.float64(0.5)}

Now we calculate the conditions:

* If stock small and and marketTrend negative, buy! Condition 1
* if stock high and marketTrend positive, sell. Condition 2 
* if stock high oder middle and marketTrend stagnation, hold. Condition 3

In [81]:
Condition1 = np.min([fS.fuzzy()["small"], fMT.fuzzy()["negative"]]) #buy
Condition2 = np.min([fS.fuzzy()["high"], fMT.fuzzy()["positive"]]) #sell
Condition3 = np.min([np.max([fS.fuzzy()["middle"], fS.fuzzy()["high"]]), fMT.fuzzy()["stagnation"]]) #hold

To obtain the activations, we create a dictionary with all relevant conditions.

In [79]:
activations = {"Buy": Condition1, "Sell": Condition2, "Hold": Condition3}
activations

{'Buy': np.float64(0.0),
 'Sell': np.float64(0.1111111111111111),
 'Hold': np.float64(0.5)}

The results depend on the conditions. Since we hold 101 Stocks we can only sell or hold 101. Note: Stocks change by decistion. The Maximum number of Stocks to buy is set to 1,000.

In [82]:
decisions = {k: int(np.round(v*1000)) if k == "Buy" else int(np.round(v*101)) for k, v in activations.items()}
decisions

{'Buy': 0, 'Sell': 11, 'Hold': 50}

Despite different conditions, sometimes reults are interdependent. In this case you can see that Sell and Hold are interconnected, because holding 50 means selling 200 - 50 = 150. Either we use the sum and we sell 11 + 50 = 61, or we use defuzzyfication.

In [85]:
avtivations = {k: v for k, v in activations.items() if k not in ["Buy"]}
avtivations

{'Sell': np.float64(0.1111111111111111), 'Hold': np.float64(0.5)}

In [86]:
resultingValues = list({k: v for k, v in decisions.items() if k not in ["Buy"]}.values())
resultingValues

[11, 50]

We can use the default method minimum value of maximal activations.

In [99]:
sell = fS.defuzzification(avtivations, resultingValues)
print("Sell", sell[0])

Sell 50


Or we use weighted mean.

In [96]:
sell = fS.defuzzification(avtivations, resultingValues, method="weightedMean")
print("Sell", int(np.round(sell)))

Sell 26
